# Cellelogistikk

## Hvordan fordeler en celle begrensede produksjonsressurser?

### Pilotprosjekt for Matematikk 1, logistikk og bioteknologi

En celle må organisere en intern forsyningskjede:

```text
næring utenfor cellen
          ↓
     transportører
          ↓
  intracellulært substrat
          ↓
 metabolske enzymer
          ↓
 proteinbyggesteiner
       ↙       ↘
  ribosomer   lipider og membran
       ↓
 nye transportører, enzymer og ribosomer
```

Det spesielle er at produksjonsapparatet må produsere nye kopier av seg selv. Ribosomene må derfor fordele sin kapasitet mellom

- nye ribosomer,
- transportproteiner,
- metabolske enzymer,
- enzymer for lipid- og membranproduksjon.

Prosjektet har fem deler:

1. **Støkiometrisk logistikk:** stoffstrømmer som et lineært system
2. **Ribosomfordeling:** ressursallokering og balansert vekst
3. **Metabolittlagre:** en trekomponents vektor-ODE
4. **Selvreplikasjon:** metabolitter og fire proteinsektorer i ett system
5. **Egenmoder:** linearisering rundt en balansert veksttilstand

### Læringsmål

Etter prosjektet skal du kunne

- bygge og tolke en støkiometrisk matrise,
- løse en stasjonær stoffbalanse,
- undersøke rang, nullrom og flaskehalser,
- formulere ribosomfordeling som et lineært system,
- skille mellom stoffstrøm, lager og produksjonskapasitet,
- skrive metabolittbalanser som en vektor-ODE,
- modellere fortynning ved cellevekst,
- simulere proteinallokering og selvreplikasjon,
- beregne en numerisk Jacobimatrise,
- tolke egenverdier og egenvektorer som responsmoder.

### Viktig avgrensning

Dette er en liten pedagogisk cellemodell. Parameterne er syntetiske undervisningsverdier, ikke data for en bestemt organisme. Modellen beskriver ikke genregulering, ATP-balanse, alle metabolitter, toksisitet, celledeling eller stokastiske molekylprosesser.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Modellens komponenter

Vi bruker tre interne stofflagre:

- $S$: intracellulært substrat
- $P$: proteinbyggesteiner
- $L$: lipid- og membranbyggesteiner

Fire proteinsektorer utfører arbeidet:

- $R$: ribosomer
- $T$: transportører
- $M$: metabolske enzymer
- $E$: lipidenzymer

Aktivitetene er:

1. transport, $v_T$
2. metabolisme, $v_M$
3. proteinsyntese, $v_R$
4. lipidsyntese, $v_L$

In [ ]:
stoff_navn = ["substrat S", "proteinbyggesteiner P", "lipider L"]
aktivitet_navn = ["transport", "metabolisme", "proteinsyntese", "lipidsyntese"]
protein_navn = ["ribosomer R", "transportører T", "metabolske enzymer M", "lipidenzymer E"]

# Del A: Støkiometrisk logistikk

## A.1 Reaksjoner og utbytte

Den forenklede kjeden er

$$S_{ute}\xrightarrow{v_T}S,$$

$$S\xrightarrow{v_M}1.8P,$$

$$P\xrightarrow{v_R}\text{protein},$$

$$P\xrightarrow{v_L}0.70L.$$

Den støkiometriske matrisen for de tre interne lagrene er

$$
\boxed{
N=
\begin{pmatrix}
1&-1&0&0\\
0&1.8&-1&-1\\
0&0&0&0.70
\end{pmatrix}.}
$$

Kolonnene svarer til aktivitetene $(v_T,v_M,v_R,v_L)$.

In [ ]:
N = np.array([
    [1.0, -1.0, 0.0, 0.0],
    [0.0, 1.8, -1.0, -1.0],
    [0.0, 0.0, 0.0, 0.70]
])

print("N =
", N)
print("Rang:", np.linalg.matrix_rank(N))

## A.2 Balansert produksjon

Anta at cellen per tidsenhet må produsere

$$d_R=0.90$$

enheter protein og

$$d_L=0.28$$

enheter lipid.

Stasjonær metabolittbalanse krever

$$v_T-v_M=0,$$

$$1.8v_M-v_R-v_L=0,$$

$$0.70v_L=d_L,$$

$$v_R=d_R.$$

Dette gir et $4\times4$-system.

## Oppgave A1: Finn stoffstrømmene

Bygg systemet $Av=b$, løs det og kontroller residualet.

In [ ]:
d_protein = 0.90
d_lipid = 0.28

A_flux = np.array([
    [1.0, -1.0, 0.0, 0.0],
    [0.0, 1.8, -1.0, -1.0],
    [0.0, 0.0, 0.0, 0.70],
    [0.0, 0.0, 1.0, 0.0]
])

b_flux = np.array([0.0, 0.0, d_lipid, d_protein])
v_flux = ...

for navn, verdi in zip(aktivitet_navn, v_flux):
    print(f"{navn:16s}: {verdi:7.4f}")
print("Residualnorm:", ...)

## Oppgave A2: Nullrom og alternative indre strømmer

Matrisen $N$ har flere aktiviteter enn interne metabolitter. Finn nullrommet numerisk med SVD.

En nullromsvektor beskriver en kombinasjon av reaksjonsstrømmer som ikke endrer de interne stofflagrene.

In [ ]:
U, s, VT = np.linalg.svd(N)
toleranse = 1e-12
rang = np.sum(s > toleranse)
nullrom = ...

print("Singulærverdier:", s)
print("Nullromsbasis:
", nullrom)
print("Kontroll N @ Z:
", ...)

## A.3 En enzymkapasitetsmatrise

Anta at hver aktivitet krever enzym- eller ribosomkapasitet:

$$
\begin{pmatrix}
1/k_T&0&0&0\\
0&1/k_M&0&0\\
0&0&1/k_R&0\\
0&0&0&1/k_L
\end{pmatrix}v
=
\begin{pmatrix}
c_T\\c_M\\c_R\\c_E
\end{pmatrix}.
$$

I denne første modellen er sammenhengen lineær.

In [ ]:
k_kapasitet = np.array([5.0, 3.0, 2.5, 1.5])
proteinbehov = v_flux/k_kapasitet

for navn, c in zip(protein_navn, proteinbehov[[2,0,1,3]]):
    print(f"{navn:22s}: {c:7.4f}")
print("Samlet proteinbehov:", np.sum(proteinbehov))

## Oppgave A3: Flaskehals og bortfall

Undersøk:

- 25 prosent lavere transportkapasitet
- 25 prosent lavere metabolsk kapasitet
- større lipidbehov
- høyere proteinbehov
- bortfall av lipidenzymet

Hvilken aktivitet krever størst proteininvestering? Når blir produksjonsplanen umulig dersom total proteinmengde er begrenset til $0.75$?

# Del B: Ribosomfordeling og balansert vekst

## B.1 Fire proteinsektorer

La

$$
\alpha=
\begin{pmatrix}
\alpha_R\\\alpha_T\\\alpha_M\\\alpha_E
\end{pmatrix}
$$

være andelen av ribosomkapasiteten som brukes på hver sektor.

$$
\boxed{\alpha_R+\alpha_T+\alpha_M+\alpha_E=1.}
$$

Ved balansert vekst må produksjonen av sektor $j$ være lik fortynningen:

$$
\boxed{\beta_j\alpha_j=\mu c_j.}
$$

Her er $c_j$ ønsket proteinandel og $\beta_j$ produksjonskapasiteten per allokeringsenhet.

In [ ]:
c_protein_mål = np.array([0.25, 0.15, 0.40, 0.20])
beta = np.array([1.00, 1.20, 1.10, 0.90])  # per døgn

## Oppgave B1: Finn allokering og vekstrate

De ukjente er

$$
(83{\alpha_R,\alpha_T,\alpha_M,\alpha_E,\mu}).
$$

Bygg et $5\times5$-system fra fire balanserte vekstligninger og sumkravet.

In [ ]:
A_alloc = np.zeros((5, 5))
b_alloc = np.zeros(5)

for j in range(4):
    A_alloc[j, j] = ...
    A_alloc[j, 4] = ...
A_alloc[4, :4] = ...
b_alloc[4] = 1.0

løsning_alloc = ...
alpha_bal = løsning_alloc[:4]
mu_bal = løsning_alloc[4]

print("Allokering:", alpha_bal)
print("Vekstrate:", mu_bal, "per døgn")
print("Doblingstid:", np.log(2)/mu_bal, "døgn")
print("Residualnorm:", ...)

## Oppgave B2: Ressursomfordeling

Sammenlign:

- næringsrikt miljø: målandel transportør $0.10$
- næringsfattig miljø: målandel transportør $0.30$
- langsommere ribosomer
- dyrere lipidenzym

Hold summen av proteinandelene lik 1. Hvordan endres $\alpha$ og $\mu$?

# Del C: Metabolittlagre som vektor-ODE

## C.1 Faste proteinsektorer

Vi begynner med faste proteinmengder

$$c_R=0.25,\quad c_T=0.15,\quad c_M=0.40,\quad c_E=0.20.$$

Reaksjonsratene er

$$
\boxed{v_T=k_Tc_T\frac{S_e}{K_T+S_e},}
$$

$$
\boxed{v_M=k_Mc_M\frac{S}{K_M+S},}
$$

$$
\boxed{v_R=k_Rc_R\frac{P}{K_R+P},}
$$

$$
\boxed{v_L=k_Lc_E\frac{P}{K_L+P}.}
$$

Tilstanden er

$$x=(S,P,L)^T.$$

In [ ]:
kT, kM, kR, kL = 6.0, 4.0, 3.2, 1.8
KT, KM, KR, KL = 1.0, 0.8, 0.5, 0.6
yP = 1.8
yL = 0.70
S_ute_normal = 6.0

cR, cT, cM, cE = c_protein_mål
mu_fast = 0.40  # per døgn, oppgitt i første dynamiske modell

## C.2 Stoffbalanser og fortynning

Når cellen vokser, fortynnes de intracellulære lagrene:

$$
\boxed{
\begin{aligned}
\dot S&=v_T-v_M-\mu S,\\
\dot P&=y_Pv_M-v_R-v_L-\mu P,\\
\dot L&=y_Lv_L-\mu L.
\end{aligned}}
$$

In [ ]:
def rater_metabolitt(x, S_ute=S_ute_normal):
    S, P, L = x
    vT = kT*cT*S_ute/(KT+S_ute)
    vM = kM*cM*S/(KM+S)
    vR = kR*cR*P/(KR+P)
    vL = kL*cE*P/(KL+P)
    return np.array([vT, vM, vR, vL])


def metabolitt_ode(t, x, S_ute=S_ute_normal):
    S, P, L = x
    vT, vM, vR, vL = rater_metabolitt(x, S_ute)
    return np.array([
        ...,
        ...,
        ...
    ])

## Oppgave C1: Euler og stasjonær tilstand

Simuler fra

$$x(0)=(0.5,0.5,0.2)^T$$

og kontroller om systemet nærmer seg en stasjonær tilstand. Bruk flere tidssteg.

In [ ]:
def euler_system(f, x0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0.0, n*dt, n+1)
    X = np.zeros((n+1, len(x0)))
    X[0] = x0
    for j in range(n):
        X[j+1] = ...
    return t, X

x0_C = np.array([0.5, 0.5, 0.2])

# Simuler for eksempel 30 døgn og plott S, P og L.

## Oppgave C2: Næringsskifte

La den eksterne substratkonsentrasjonen falle fra $6.0$ til $0.7$ etter fem døgn.

Undersøk:

- hvilket lager som reagerer først
- om proteinbyggesteinene blir flaskehals
- hvordan lipidproduksjonen påvirkes
- om faste proteinsektorer er en rimelig antakelse etter skiftet

In [ ]:
def S_ute_tid(t):
    return 6.0 if t < 5.0 else 0.7


def metabolitt_ode_skift(t, x):
    return metabolitt_ode(t, x, S_ute_tid(t))

# Del D: Metabolitter og selvreplikerende proteinsektorer

## D.1 Syv tilstander

Tilstanden er

$$
\boxed{X=(S,P,L,c_R,c_T,c_M,c_E)^T.}
$$

Proteinproduksjonen fordeles med $\alpha$:

$$
\boxed{\dot c_j=\alpha_jv_R-\mu c_j.}
$$

Vi definerer vekstraten som total proteinsyntese per enhet samlet protein:

$$
\boxed{\mu=\frac{v_R}{c_R+c_T+c_M+c_E}.}
$$

Dermed holdes samlet proteinmengde konstant i denne ressursfordelingsmodellen, mens sammensetningen kan endres.

## D.2 En enkel allokeringspolitikk

Ved lite næring prioriterer cellen transportører. Ved mye næring prioriteres ribosomer og metabolisme.

Vi bruker en glatt blanding mellom to allokeringsvektorer:

$$
\alpha_{fattig}=(0.20,0.35,0.30,0.15)^T,
$$

$$
\alpha_{rik}=(0.32,0.10,0.42,0.16)^T.
$$

Blandingsfaktoren er

$$w(S_e)=\frac{S_e}{K_a+S_e},$$

$$
\boxed{\alpha=(1-w)\alpha_{fattig}+w\alpha_{rik}.}
$$

In [ ]:
alpha_fattig = np.array([0.20, 0.35, 0.30, 0.15])
alpha_rik = np.array([0.32, 0.10, 0.42, 0.16])
K_allokering = 1.5


def alpha_policy(S_ute):
    w = S_ute/(K_allokering+S_ute)
    alpha = (1-w)*alpha_fattig+w*alpha_rik
    return alpha/np.sum(alpha)

## Oppgave D1: Implementer selvreplikasjonsmodellen

Bruk de samme rateuttrykkene som i del C, men la proteinmengdene være tilstander. Bruk allokeringspolitikken over.

In [ ]:
def celle_ode(t, X, S_ute_funksjon=lambda t: 6.0):
    S, P, L, cR, cT, cM, cE = X
    S_ute = S_ute_funksjon(t)

    vT = kT*cT*S_ute/(KT+S_ute)
    vM = kM*cM*S/(KM+S)
    vR = kR*cR*P/(KR+P)
    vL = kL*cE*P/(KL+P)

    c_sum = cR+cT+cM+cE
    mu = ...
    alpha = alpha_policy(S_ute)

    dS = ...
    dP = ...
    dL = ...
    dc = ...

    return np.concatenate([[dS, dP, dL], dc])

## Oppgave D2: Skifte fra rikt til fattig miljø

Start med proteinfordelingen $c_{protein,mål}$ og la næringen falle etter ti døgn.

Plott:

- metabolittlagrene
- de fire proteinsektorene
- allokeringsandelene
- vekstraten

Forklar hvorfor transportørsektoren øker, og hvorfor tilpasningen ikke skjer momentant.

In [ ]:
X0_D = np.concatenate([[0.8, 0.8, 0.3], c_protein_mål])


def S_ute_D(t):
    return 6.0 if t < 10.0 else 0.7

# Simuler for eksempel 60 døgn.

## Oppgave D3: Produksjonsapparatet produserer seg selv

Undersøk:

- 50 prosent færre ribosomer i starttilstanden
- 50 prosent færre transportører
- 50 prosent færre metabolske enzymer
- langsommere ribosomer

Hvilken forstyrrelse gir lengst gjenopprettingstid? Kan cellen bli fanget i en tilstand med svært lav vekst?

# Del E: Linearisering og egenmoder

## E.1 Numerisk Jacobimatrise

For en tilstand $X^*$ kan Jacobimatrisen tilnærmes med sentrale differanser:

$$
\boxed{
J_{ij}\approx
\frac{f_i(X^*+\varepsilon e_j)-f_i(X^*-\varepsilon e_j)}{2\varepsilon}.}
$$

Vi finner først en tilnærmet likevekt ved å simulere lenge i konstant rikt miljø.

In [ ]:
def numerisk_jacobi(f, x, eps=1e-6):
    x = np.asarray(x, dtype=float)
    n = len(x)
    J = np.zeros((n, n))
    for j in range(n):
        e = np.zeros(n)
        e[j] = eps
        J[:, j] = ...
    return J

## Oppgave E1: Finn en balansert veksttilstand

Simuler modellen med konstant $S_e=6.0$ til endringene er små. Kall sluttverdien $X^*$ og kontroller $\|f(X^*)\|$.

In [ ]:
# Finn X_stjerne ved lang simulering med lite tidssteg.
# Kontroller normen til celle_ode(0, X_stjerne).

## Oppgave E2: Egenverdier og biologiske modi

Beregn egenverdier og høyre egenvektorer til $J$.

- Negative realdeler betyr at små forstyrrelser dempes.
- Komplekse egenverdier betyr dempede oscillasjoner.
- Egenverdien nærmest null gir den langsomste tilpasningen.

Undersøk komponentene i den langsomste egenvektoren. Er moden dominert av metabolittlager, transportører, ribosomer eller metabolsk kapasitet?

In [ ]:
# J = numerisk_jacobi(...)
# egenverdier, P = np.linalg.eig(J)
# Sorter etter realdel og tolk langsomste stabile mode.

## E.3 Direkte og lineær respons

For en liten forstyrrelse $\xi=X-X^*$ gjelder omtrent

$$
\boxed{\dot\xi=J\xi.}
$$

Sammenlign:

- det ikke-lineære systemet
- den lineære modellen
- en modal løsning dersom $J$ er diagonaliserbar

Bruk først en liten forstyrrelse. Øk deretter forstyrrelsen og observer når lineariseringen blir dårlig.

# Fordypning: To alternative metabolske ruter

Cellen kan ha to ruter fra substrat til proteinbyggesteiner:

## Rute 1: Materialeffektiv

- høyt utbytte: $1.8P$ per $S$
- lavere enzymomsetning

## Rute 2: Katalytisk effektiv

- lavere utbytte: $1.2P$ per $S$
- høyere enzymomsetning

Dette gir en avveining:

$$
\boxed{
\text{godt substratutbytte}
\quad\text{mot}\quad
\text{høy gjennomstrømning per enzym}.}
$$

Studentene kan innføre to metabolske enzymsektorer og undersøke hvilken rute som er fordelaktig ved lav og høy substratkonsentrasjon.

# Modellkritikk

Diskuter minst seks punkter:

- Bare tre metabolittlagre er med.
- Energi og ATP modelleres ikke eksplisitt.
- Reaksjonsutbyttene er konstante.
- Enzymkinetikken er svært forenklet.
- Proteinsektorene er aggregater av mange ulike proteiner.
- Ribosomallokeringen bestemmes av en oppgitt politikk, ikke genregulering.
- Cellen har ingen eksplisitt DNA-, RNA- eller transkripsjonsmodell.
- Samlet proteinmengde holdes konstant i selvreplikasjonsmodellen.
- Lipidlageret representerer ikke cellegeometri direkte.
- Vekstraten er definert fra proteinsyntesen alene.
- Celledeling er ikke en diskret hendelse.
- Modellen er deterministisk.
- Parameterne er syntetiske.
- Jacobimatrisen avhenger av valgt likevekt og numerisk differansesteg.
- Den lineære modellen gjelder bare nær likevekten.

## Mulige videreføringer

- eksplisitt ATP-balanse
- flere metabolske ruter
- membranareal og cellevolum
- transkripsjon og mRNA
- celledeling
- stokastisk genuttrykk
- optimal ribosomallokering
- sammenligning med målte vekstlover
- kobling til biohydrogen eller bioreaktor

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan reaksjonene ga en støkiometrisk matrise,
2. hvordan stasjonære stoffstrømmer ble funnet,
3. hva nullrommet betyr fysisk,
4. hvordan enzymkapasitet ga et logistisk flaskehalsproblem,
5. hvordan ribosomfordelingen ble et lineært system,
6. hvordan metabolittlagrene ga en vektor-ODE,
7. hvorfor vekst gir fortynningsledd,
8. hvordan proteinsektorene tilpasset seg et næringsskifte,
9. hva egenverdiene fortalte om tilpasningshastigheter,
10. hvorfor produksjonsapparatet må bruke kapasitet på å produsere seg selv.

## Faglig bakgrunn

Prosjektet er inspirert av ressursallokeringsmodeller for selvreplikerende celler. Slike modeller kombinerer støkiometriske balanser, enzymkapasiteter, ribosomallokering, membrankrav og balansert vekst. Hovedprosjektet bruker en vesentlig mindre modell slik at lineær algebra og ODE-er forblir synlige.